# Plotting Raw EEG Trials from MOABB

**Dataset**: BNCI2014-001 (Motor Imagery)
**Subject**: 1
**Paradigm**: MotorImagery (n_classes=2)
**Channels**: 22 EEG
**Sampling rate**: 250 Hz

---

## Overview

This notebook loads motor imagery epochs from BNCI2014-001 and visualizes raw EEG trials for left-hand and right-hand imagery, showing all 22 channels with vertical offsets for the first 5 seconds.

## What this notebook does

- Loads BNCI2014-001 subject 1 via MOABB
- Extracts epochs with the MotorImagery paradigm
- Selects one left-hand and one right-hand trial
- Plots both trials as multi-channel traces with offsets

## What you should expect to see

- Two panels: left-hand and right-hand motor imagery
- 22 channel traces stacked vertically with offsets
- 5 seconds of data per trial
- Distinctive patterns over motor cortex channels (C3, C4, Cz)

## Key parameters

| Parameter | Value | Meaning |
| --- | --- | --- |
| dataset | BNCI2014_001 | MOABB motor imagery dataset |
| subjects | [1] | Subject 1 only |
| n_classes | 2 | MotorImagery paradigm classes |
| plot_duration | 5 s | Seconds of data to display |
| offset | auto | Vertical spacing between channels |



## 1. Install dependencies


In [ ]:
!pip install moabb mne scipy numpy plotly


## 2. Load MOABB dataset

MOABB downloads data automatically on first use (~44 MB for subject 1). Subsequent runs use cached data.



In [ ]:
from moabb.datasets import BNCI2014_001
ds = BNCI2014_001()
sessions = ds.get_data(subjects=[1])
subject_key = list(sessions.keys())[0]
session_dict = sessions[subject_key]
n_sessions = len(session_dict)
n_runs = len(next(iter(session_dict.values())))
first_run = next(iter(next(iter(session_dict.values())).values()))
n_channels_raw = len(first_run.ch_names)
sfreq = first_run.info['sfreq']
print(f'Subject 1: {n_sessions} sessions, {n_runs} runs/session')
print(f'Raw channels: {n_channels_raw}, Sampling rate: {sfreq} Hz')
print(f'Channel names: {first_run.ch_names}')



In [ ]:
from moabb.paradigms import MotorImagery
paradigm = MotorImagery(n_classes=2)
X, labels, meta = paradigm.get_data(dataset=ds, subjects=[1])
print(f'X shape: {X.shape}  (n_trials, n_channels, n_samples)')
print(f'Labels shape: {labels.shape}')
print(f'Meta shape: {meta.shape}')



## 3. Explore the data

We print the epoch shapes and available labels.


In [ ]:
import numpy as np
unique_labels, counts = np.unique(labels, return_counts=True)
print(f'Epoch channels: {X.shape[1]}')
print(f'Epoch samples: {X.shape[2]}')
print(f'Epoch duration: {X.shape[2] / sfreq:.2f} s')
print(f'Unique labels: {list(unique_labels)}')
print(f'Trials per class: {dict(zip(unique_labels, counts))}')
print(f'Total trials: {X.shape[0]}')



## 4. Apply the analysis

We select one left-hand and one right-hand trial and prepare the time axis for plotting.



In [ ]:
import numpy as np
sfreq = 250
n_samples = X.shape[2]
plot_samples = min(int(5 * sfreq), n_samples)
time = np.arange(plot_samples) / sfreq
left_idx = np.where(labels == 'left_hand')[0][0]
right_idx = np.where(labels == 'right_hand')[0][0]
left_trial = X[left_idx, :, :plot_samples]
right_trial = X[right_idx, :, :plot_samples]
print(f'Left hand trial index: {left_idx}')
print(f'Right hand trial index: {right_idx}')
print(f'Plotting {plot_samples} samples ({plot_samples/sfreq:.1f} s)')



## 5. Interactive plot

**What to look for:**

- Two panels showing left-hand and right-hand motor imagery
- 22 channels stacked with vertical offsets
- Differences in amplitude patterns between the two conditions
- Motor cortex channels (C3, C4) may show condition-specific activity




In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
n_channels = X.shape[1]
offset_step = 1.2 * np.max(np.abs(X))
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=('Left hand motor imagery', 'Right hand motor imagery'))
for ch in range(n_channels):
    fig.add_trace(go.Scatter(x=time, y=left_trial[ch] + ch * offset_step,
                             mode='lines', line=dict(width=0.6),
                             showlegend=False), row=1, col=1)
for ch in range(n_channels):
    fig.add_trace(go.Scatter(x=time, y=right_trial[ch] + ch * offset_step,
                             mode='lines', line=dict(width=0.6),
                             showlegend=False), row=2, col=1)
fig.update_xaxes(title_text='Time (s)', row=2, col=1)
fig.update_layout(height=800, title_text='MOABB BNCI2014-001 - Motor Imagery Trials',
                  xaxis_range=[0, plot_samples/sfreq])
fig.show()



## What did we learn?

- MOABB epochs contain multi-channel EEG trials ready for visualization
- Left-hand and right-hand motor imagery show distinct channel patterns
- Vertical offsets make it easy to inspect all 22 channels simultaneously
- The motor cortex channels (C3, C4) are key for motor imagery classification


